<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-05-llm-api-engineering/lesson-5.5-function-calling-and-tools/notebooks/exercises-5.5.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 5.5 — Function Calling & Tools
Netsetos GenAI Course v2.0 | Module 5

Tool definitions, agent loops, MCP, streaming, safety, India tools


In [ ]:
# pip install openai anthropic google-generativeai -q
print('Ready for function calling & tools')


## Ex 1: OpenAI Function Calling


In [ ]:
from openai import OpenAI
import json

client = OpenAI()

tools = [{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': 'Get current weather for a city',
        'parameters': {
            'type': 'object',
            'properties': {
                'city': {'type': 'string', 'description': 'City name'},
                'unit': {'type': 'string', 'enum': ['celsius', 'fahrenheit']}
            },
            'required': ['city']
        }
    }
}]

resp = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': 'Weather in Hyderabad?'}],
    tools=tools
)

tc = resp.choices[0].message.tool_calls[0]
print(f'Function: {tc.function.name}')
print(f'Args: {tc.function.arguments}')
print(f'ID: {tc.id}')


## Ex 2: tool_choice Modes


In [ ]:
# Test all 4 modes
for mode in ['auto', 'required', 'none']:
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': 'Hello!'}],
        tools=tools,
        tool_choice=mode
    )
    has_tools = bool(resp.choices[0].message.tool_calls)
    print(f'{mode:10s}: tool_calls={has_tools}')

# Force specific tool
resp = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': 'Hello!'}],
    tools=tools,
    tool_choice={'type': 'function', 'function': {'name': 'get_weather'}}
)
print(f'specific:   tool_calls={bool(resp.choices[0].message.tool_calls)}')


## Ex 3: Multi-Step Tool Execution


In [ ]:
def execute_tool(name, args_json):
    args = json.loads(args_json)
    if name == 'get_weather':
        return f"Weather in {args['city']}: 32°C, sunny"
    elif name == 'calculate':
        return str(eval(args['expression']))  # Demo only! Never eval in prod
    return 'Unknown tool'

# Full round-trip
messages = [{'role': 'user', 'content': 'Weather in Hyderabad?'}]
resp = client.chat.completions.create(model='gpt-4o-mini', messages=messages, tools=tools)

# Append assistant message with tool call
messages.append(resp.choices[0].message)

# Execute and append tool result
for tc in resp.choices[0].message.tool_calls:
    result = execute_tool(tc.function.name, tc.function.arguments)
    messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': result})

# Get final response
final = client.chat.completions.create(model='gpt-4o-mini', messages=messages, tools=tools)
print(f'Final: {final.choices[0].message.content}')


## Ex 4: Agent Loop with Termination


In [ ]:
def agent_loop(user_msg, tools, max_iter=10, max_tokens=50000):
    messages = [{'role': 'system', 'content': 'Use tools to answer questions.'},
                {'role': 'user', 'content': user_msg}]
    total_tokens = 0
    
    for i in range(max_iter):
        # Disable tools on last iteration to force text answer
        current_tools = tools if i < max_iter - 1 else []
        
        resp = client.chat.completions.create(
            model='gpt-4o-mini', messages=messages, tools=current_tools or None)
        
        total_tokens += resp.usage.total_tokens
        msg = resp.choices[0].message
        messages.append(msg)
        
        # Natural termination: no tool calls
        if not msg.tool_calls:
            print(f'Done in {i+1} iterations, {total_tokens} tokens')
            return msg.content
        
        # Execute tools
        for tc in msg.tool_calls:
            result = execute_tool(tc.function.name, tc.function.arguments)
            messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': result})
        
        # Token budget check
        if total_tokens > max_tokens:
            print(f'Token budget exceeded: {total_tokens}')
            break
    
    # Fallback: force text answer
    resp = client.chat.completions.create(model='gpt-4o-mini', messages=messages)
    return resp.choices[0].message.content

# result = agent_loop('Weather in Hyderabad?', tools)
print('Agent loop defined with 5 termination conditions')


## Ex 5: Streaming Tool Calls


In [ ]:
# Accumulate streaming tool call arguments
tool_buffers = {}  # index -> {id, name, arguments}

stream = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': 'Weather in Delhi and Mumbai?'}],
    tools=tools, stream=True
)

for chunk in stream:
    delta = chunk.choices[0].delta
    if delta.tool_calls:
        for tc in delta.tool_calls:
            idx = tc.index
            if idx not in tool_buffers:
                tool_buffers[idx] = {'id': tc.id, 'name': tc.function.name, 'args': ''}
            if tc.function.arguments:
                tool_buffers[idx]['args'] += tc.function.arguments

for idx, buf in sorted(tool_buffers.items()):
    args = json.loads(buf['args'])
    print(f'Tool {idx}: {buf["name"]}({args})')


## Ex 6: MCP Server (FastMCP)


In [ ]:
# MCP server in Python
mcp_code = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("gst-tools")

@mcp.tool()
async def validate_gstin(gstin: str) -> dict:
    """Validate an Indian GST Identification Number."""
    import re
    pattern = r"^[0-9]{2}[A-Z]{5}[0-9]{4}[A-Z]{1}[1-9A-Z]{1}Z[0-9A-Z]{1}$"
    valid = bool(re.match(pattern, gstin))
    return {"gstin": gstin, "valid": valid, "state_code": gstin[:2] if valid else None}

@mcp.tool()
async def calculate_gst(amount: float, rate: float = 18.0) -> dict:
    """Calculate GST for a given amount."""
    gst = amount * rate / 100
    return {"base": amount, "gst": round(gst, 2), "total": round(amount + gst, 2)}

mcp.run(transport="stdio")
'''
print(mcp_code)
print('Run: python gst_server.py')
print('Test: npx @modelcontextprotocol/inspector python gst_server.py')


## Ex 7: Safety — Permission Tiers


In [ ]:
from enum import Enum

class ToolTier(Enum):
    SAFE = 'safe'        # Auto-approve (search, read, list)
    WRITE = 'write'      # Requires user approval (update, create)
    DANGEROUS = 'danger'  # Restricted + approval (delete, pay, send)

TOOL_PERMISSIONS = {
    'search_web': ToolTier.SAFE,
    'read_file': ToolTier.SAFE,
    'write_file': ToolTier.WRITE,
    'send_email': ToolTier.WRITE,
    'delete_file': ToolTier.DANGEROUS,
    'make_payment': ToolTier.DANGEROUS,
}

def check_permission(tool_name: str, user_role: str = 'user') -> bool:
    tier = TOOL_PERMISSIONS.get(tool_name, ToolTier.DANGEROUS)
    if tier == ToolTier.SAFE:
        return True  # Auto-approve
    elif tier == ToolTier.WRITE:
        confirm = input(f'Approve {tool_name}? (y/n): ')
        return confirm.lower() == 'y'
    elif tier == ToolTier.DANGEROUS:
        if user_role != 'admin':
            print(f'BLOCKED: {tool_name} requires admin role')
            return False
        confirm = input(f'⚠️ DANGEROUS: Approve {tool_name}? (y/n): ')
        return confirm.lower() == 'y'
    return False

for tool in TOOL_PERMISSIONS:
    print(f'{tool:20s} → {TOOL_PERMISSIONS[tool].value}')


## Ex 8: India Stack Tool Definitions


In [ ]:
india_tools = [
    {
        'type': 'function',
        'function': {
            'name': 'validate_gstin',
            'description': 'Validate Indian GSTIN and return business details',
            'parameters': {
                'type': 'object',
                'properties': {
                    'gstin': {'type': 'string', 'pattern': '^[0-9]{2}[A-Z]{5}[0-9]{4}[A-Z]{1}[1-9A-Z]{1}Z[0-9A-Z]{1}$'}
                },
                'required': ['gstin']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'create_upi_payment',
            'description': 'Create a UPI payment request (requires user approval)',
            'parameters': {
                'type': 'object',
                'properties': {
                    'vpa': {'type': 'string', 'description': 'UPI VPA (e.g., user@upi)'},
                    'amount': {'type': 'number', 'description': 'Amount in INR'},
                    'description': {'type': 'string'}
                },
                'required': ['vpa', 'amount']
            }
        }
    }
]

print(f'Defined {len(india_tools)} India Stack tools')
for t in india_tools:
    print(f'  {t["function"]["name"]}: {t["function"]["description"]}')


---
## Recap
Function calling gives LLMs real-world capabilities via structured tool invocations. tool_choice (auto/required/specific/none) controls when and which tools are called. The canonical agent loop is a while loop with 5 termination conditions. MCP (10K+ servers, 97M downloads) unified tool integration as the industry standard. Streaming tool calls accumulate partial JSON by index. Production safety requires defense in depth: human-in-the-loop, sandboxing, permission tiers, rate limiting, input validation, OWASP compliance. India Stack (UPI, Aadhaar, GST, AA, ONDC) + Sarvam AI + WhatsApp = the Indian agent platform.
